# Bandarlog · generador de ilustraciones en Colab gratuito
Ejecuta las celdas en orden después de incorporar la PR a `main`. Funciona con CPU; no requiere contratar Colab Pro ni una API de pago. Google limita la duración y disponibilidad de sus sesiones gratuitas. Si la sesión termina, el generador deja de estar disponible y el enlace temporal cambia al reiniciarlo.

La aplicación web y Neon siguen en Railway. Este cuaderno solo genera las ilustraciones de comisiones. No se envían nombres de personas, tareas ni fotos de perfil al modelo. No guardes claves en celdas, salidas ni GitHub.


In [ ]:
from pathlib import Path
import subprocess, sys, os
assert sys.version_info >= (3, 12), 'Selecciona un entorno Colab con Python 3.12 o superior.'
repo = Path('/content/bandarlog')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nunudiewillump/bandarlog.git', str(repo)], check=True)
assert (repo / 'ai-worker/model.py').exists(), 'Primero hay que incorporar la PR del generador a main.'
venv = Path('/content/bandarlog-ai-venv')
if not venv.exists(): subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
python = str(venv / 'bin/python')
subprocess.run([python, '-m', 'pip', 'install', '-r', str(repo / 'ai-worker/requirements.txt')], check=True)


## Clave privada y arranque
Elige una clave aleatoria larga en tu gestor de contraseñas. Introdúcela abajo y guárdala también en Railway como `ARTWORK_WORKER_TOKEN`. El campo no muestra ni guarda la clave en la salida del cuaderno.


In [ ]:
from getpass import getpass
import time, urllib.request, json
worker_token = getpass('Clave privada del generador (mínimo 24 caracteres): ')
assert len(worker_token) >= 24, 'Utiliza una clave aleatoria de al menos 24 caracteres.'
worker_env = {**os.environ, 'ARTWORK_WORKER_TOKEN': worker_token, 'HF_HOME': '/content/bandarlog-model-cache', 'U2NET_HOME': '/content/bandarlog-model-cache/u2net', 'HF_HUB_DISABLE_XET': '1', 'AI_THREADS': '2'}
worker_log = open('/content/bandarlog-worker.log', 'w')
worker = subprocess.Popen([python, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7860', '--no-access-log'], cwd=str(repo / 'ai-worker'), env=worker_env, stdout=worker_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    try:
        health = json.load(urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=3))
        if health.get('ok') and health.get('configured'): break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('El generador no ha arrancado. Revisa el registro local sin compartir claves.')
print('Generador preparado; la clave no se muestra.')


## Enlace HTTPS temporal para Railway
Se abre un túnel de prueba gratuito de Cloudflare. El generador sigue protegido por la clave privada. Solo se muestra la dirección pública; la descarga del ejecutable comprueba el SHA-256 publicado por GitHub.


In [ ]:
import hashlib, re
release = json.load(urllib.request.urlopen('https://api.github.com/repos/cloudflare/cloudflared/releases/latest'))
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
digest = asset.get('digest', '')
assert digest.startswith('sha256:'), 'Falta el SHA-256 oficial del ejecutable; no se ejecutará la descarga.'
data = urllib.request.urlopen(asset['browser_download_url']).read()
assert hashlib.sha256(data).hexdigest() == digest.split(':', 1)[1], 'La verificación SHA-256 ha fallado.'
cloudflared = Path('/content/bandarlog-cloudflared')
cloudflared.write_bytes(data)
cloudflared.chmod(0o700)
tunnel_log_path = Path('/content/bandarlog-tunnel.log')
tunnel_log = tunnel_log_path.open('w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log_path.read_text())
    if matches:
        worker_url = matches[-1]
        break
    time.sleep(1)
else:
    raise RuntimeError('No se ha obtenido un enlace temporal. El proveedor puede no estar disponible.')
print('En Railway, establece ARTWORK_WORKER_URL en:')
print(worker_url)
print('ARTWORK_WORKER_TOKEN debe contener la clave que has introducido, sin compartirla aquí.')


## Prueba opcional de generación
La primera ejecución descarga los modelos y puede tardar varios minutos. Los siguientes resultados se generan en la CPU gratuita de esta sesión. Las imágenes se muestran abajo; no se envía correo ni se modifica la base de datos de producción.


In [ ]:
import base64, io
from PIL import Image
from IPython.display import display
request = urllib.request.Request(worker_url + '/generate', data=json.dumps({'name':'Cocina','description':'Menús del campamento'}).encode(), headers={'Content-Type':'application/json','Authorization':'Bearer '+worker_token}, method='POST')
with urllib.request.urlopen(request, timeout=600) as response:
    artwork = json.load(response)
for key in ['iconUrl','mascotUrl']:
    display(Image.open(io.BytesIO(base64.b64decode(artwork[key].split(',',1)[1]))))
print('Generación comprobada. Puedes crear una comisión en Bandarlog con esta sesión abierta.')


## Detener la prueba
Al terminar, detén los procesos y elimina `ARTWORK_WORKER_URL` de Railway si no vas a mantener esta sesión. Las ilustraciones ya guardadas en Neon no se pierden cuando Colab se desconecta.


In [ ]:
for process in [globals().get('tunnel'), globals().get('worker')]:
    if process is not None and process.poll() is None: process.terminate()
worker_token = ''
print('Generador y túnel detenidos.')
